# ⚡ City-ANPR: Re-Evaluate Fine-Tuned License Plate OCR (No Training)

Welcome to the **Fine-Tuned Plate OCR Evaluation** notebook! In this notebook, we evaluate the fine-tuned **`fast-plate-ocr` (`cct-s-v2-global-model`)** checkpoint saved in Google Drive from our fine-tuning run:
`/content/drive/MyDrive/anpr/ocr_runs/finetune/2026-10-03_15-58-30/`

---

### 🔍 Background & The `PlatePrediction` Object Fix
In the initial fine-tuning run, predictions showed **0% exact match** despite successful model convergence. The root cause:
- In `fast-plate-ocr`, `LicensePlateRecognizer.run()` returns a `list[PlatePrediction]`, where each element is an instance of `PlatePrediction` (`@dataclass(frozen=True)`).
- The decoded string is stored in the **`.plate` attribute** (`prediction.plate`).
- Because the previous evaluation code checked `hasattr(result, 'plate')` directly on the returned `list` (which is `False`), it fell back to `str(result)`: `"[PlatePrediction(plate='MH20EE045', char_probs=None, ...)]"`.
- Stripping non-alphanumeric characters turned this into `"PLATEPREDICTIONPLATEMH20EE045..."`, which never matched the true plate `"MH20EE045"`!
- **The Fix**: We properly extract `preds[0].plate` from the prediction object, strip library padding characters, and normalize to uppercase alphanumeric.

---

### 🎯 Evaluation Pipeline Overview (No Training):
1. **Environment Setup & Data Extraction**: Mount Drive, copy `processed_for_colab.zip` to Colab SSD, run `!rm -rf /content/data` first, unzip cleanly, install `fast-plate-ocr[onnx]`, and load `ocr/real/labels.csv`.
2. **Load Fine-Tuned Model**: Load the fine-tuned model from the run folder via **ONNX Runtime** (the supported inference path in `fast-plate-ocr`).
3. **Pre-Metric Sanity Check**: Run inference on 10 validation crops and assert that predictions are valid alphanumeric plate strings (only A-Z, 0-9, length < 13, and no `PlatePrediction` wrapper strings).
4. **Validation Evaluation**: Evaluate on VAL (raw vs Indian syntax post-processing) and dynamically select the superior variant for the test split.
5. **Held-Out Test Evaluation**: Run the held-out TEST split **strictly ONCE** across 5 reporting slices (all crops, unseen text, unique plates, clean plates, suspicious plates).
6. **Quality Tiers**: Measure performance across 3 resolution tiers (<24 px, 24-40 px, >40 px height).
7. **Multi-Frame Sequence Voting (Estimate)**: Combine multi-frame video tracks via character-level majority voting.
8. **Error Diagnostics**: Extract top-15 character confusions and display a 6x5 visual grid of the 30 worst remaining errors.
9. **Export & Honest Benchmark Comparison**: Export `metrics.json` (Python native types), save summary tables, generate the honest comparison table using **ONLY measured project numbers**, and print a fully dynamic summary.

> 💡 **Beginner Tip**: This notebook executes in 2–3 minutes on Colab CPU or T4 GPU runtime. Press **Runtime > Run all** or execute cells sequentially.

## Step 1: Environment Setup, Local SSD Copy & fast-plate-ocr Installation

In this step, we prepare our evaluation environment:
1. Mount Google Drive.
2. Copy `processed_for_colab.zip` to `/content/` local SSD.
3. Cleanly remove any existing `/content/data` directory (`!rm -rf /content/data`) and unzip the fresh archive.
4. Install `fast-plate-ocr[onnx]` along with standard computer vision dependencies.
5. Load `ocr/real/labels.csv` and verify image counts per split.

In [ ]:
# Step 1: Mount Drive, copy archive to Colab SSD, clean & unzip, install fast-plate-ocr
import os
import sys
import shutil
import zipfile
from pathlib import Path
import pandas as pd

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[OK] Google Drive mounted.")
except Exception as e:
    print(f"[NOTE] Drive mount skipped or already mounted: {e}")

# 2. Copy zip from Drive to local SSD
DRIVE_ZIP = Path('/content/drive/MyDrive/anpr/processed_for_colab.zip')
LOCAL_ZIP = Path('/content/processed_for_colab.zip')

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f"Dataset archive not found at {DRIVE_ZIP}! Please ensure processed_for_colab.zip is in Google Drive under anpr/.")

if not LOCAL_ZIP.exists():
    print(f"Copying {DRIVE_ZIP.name} ({DRIVE_ZIP.stat().st_size / (1024*1024):.1f} MB) to local SSD /content/...")
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
    print("[OK] Archive copied to local SSD.")
else:
    print("[OK] Archive already present on local SSD.")

# 3. Remove /content/data first as instructed, then unzip cleanly
DATA_DIR = Path('/content/data')
if DATA_DIR.exists():
    print("Cleaning up existing /content/data directory (!rm -rf /content/data)...")
    shutil.rmtree(DATA_DIR)

print(f"Unzipping {LOCAL_ZIP.name} into /content/data...")
with zipfile.ZipFile(LOCAL_ZIP, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)
print("[OK] Dataset unzipped successfully.")

# 4. Install fast-plate-ocr with ONNX runtime support
print("Installing fast-plate-ocr[onnx]...")
!pip install -q "fast-plate-ocr[onnx]"

# 5. Load and inspect real OCR labels
LABELS_CSV = DATA_DIR / 'processed' / 'ocr' / 'real' / 'labels.csv'
if not LABELS_CSV.exists():
    # Fallback path if unzipped directly without 'processed' wrapper
    alt_csv = DATA_DIR / 'ocr' / 'real' / 'labels.csv'
    if alt_csv.exists():
        LABELS_CSV = alt_csv
    else:
        raise FileNotFoundError(f"labels.csv not found at {LABELS_CSV} or {alt_csv}!")

df_real = pd.read_csv(LABELS_CSV)
# Derive split column from path (e.g. real/train/... -> train)
df_real['split'] = df_real['path'].apply(lambda p: str(p).replace('\\', '/').split('/')[1])

print("=" * 75)
print(f"=== Loaded Real OCR Dataset: {len(df_real)} Total Crops ===")
print("=" * 75)
for s in ['train', 'val', 'test']:
    sub = df_real[df_real['split'] == s]
    susp = sub['suspicious'].sum() if 'suspicious' in sub.columns else 0
    unseen = (~sub['text_seen_in_train']).sum() if 'text_seen_in_train' in sub.columns else len(sub)
    print(f"  - {s.upper():<5}: {len(sub):>4} crops | {susp:>2} suspicious | {unseen:>3} unseen text")
print("=" * 75)


## Step 2: Load the Fine-Tuned Model from the Google Drive Run Folder

### ℹ️ Supported Inference Path in `fast-plate-ocr`
- **ONNX Runtime is the Supported Inference Path**: `fast-plate-ocr` provides the `LicensePlateRecognizer` class for inference, which executes optimized ONNX graphs using `onnxruntime.InferenceSession`.
- The `.keras` file (`best.keras`) is a training checkpoint in Keras 3 format. `LicensePlateRecognizer` does **not** load `.keras` files directly—it requires an exported **`.onnx` model** (`best.onnx`) and its corresponding **`PlateConfig` YAML** (`cct_s_v2_global_plate_config.yaml`).
- In this cell, we look for the exported ONNX model in `/content/drive/MyDrive/anpr/ocr_runs/finetune/2026-10-03_15-58-30/`. If `best.keras` is found but has not yet been exported to ONNX, we automatically run `fast-plate-ocr export` on the fly to generate it.

In [ ]:
# Step 2: Load the fine-tuned model and config from the specified run folder
import re
import shutil
from pathlib import Path
from fast_plate_ocr import LicensePlateRecognizer

# 1. Target run directory specified by the user
SPECIFIED_RUN_DIR = Path('/content/drive/MyDrive/anpr/ocr_runs/finetune/2026-10-03_15-58-30')
BASE_FINETUNE_DIR = Path('/content/drive/MyDrive/anpr/ocr_runs/finetune')

if SPECIFIED_RUN_DIR.exists():
    RUN_DIR = SPECIFIED_RUN_DIR
    print(f"[FOUND] Target run directory: {RUN_DIR}")
else:
    # Fallback to the latest timestamped run folder if directory name differs
    run_dirs = sorted([d for d in BASE_FINETUNE_DIR.glob('*') if d.is_dir() and d.name not in ('eval_fixed',)], key=lambda p: p.stat().st_mtime, reverse=True)
    if run_dirs:
        RUN_DIR = run_dirs[0]
        print(f"[NOTE] Specified folder {SPECIFIED_RUN_DIR} not found. Using latest found run directory: {RUN_DIR}")
    else:
        raise FileNotFoundError(f"No run folders found under {BASE_FINETUNE_DIR}!")

# 2. Locate plate config YAML
config_candidates = [
    RUN_DIR / 'cct_s_v2_global_plate_config.yaml',
    BASE_FINETUNE_DIR / 'cct_s_v2_global_plate_config.yaml',
    Path('/content/cct_s_v2_global_plate_config.yaml'),
]
PLATE_CONFIG_PATH = None
for cp in config_candidates:
    if cp.exists():
        PLATE_CONFIG_PATH = cp
        break

if PLATE_CONFIG_PATH is None:
    print("Plate config not found locally. Downloading official cct_s_v2_global_plate_config.yaml...")
    !wget -q -O /content/cct_s_v2_global_plate_config.yaml https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates/cct_s_v2_global_plate_config.yaml
    PLATE_CONFIG_PATH = Path('/content/cct_s_v2_global_plate_config.yaml')

print(f"Using Plate Config: {PLATE_CONFIG_PATH}")

# 3. Locate or export ONNX model
# Supported inference format: ONNX (.onnx). We search for existing ONNX or export from best.keras.
onnx_candidates = [
    RUN_DIR / 'best.onnx',
    RUN_DIR / 'cct_s_v2_finetuned_indian.onnx',
    BASE_FINETUNE_DIR / 'cct_s_v2_finetuned_indian.onnx',
] + list(RUN_DIR.glob('*.onnx'))

ONNX_MODEL_PATH = None
for op in onnx_candidates:
    if op.exists():
        ONNX_MODEL_PATH = op
        break

if ONNX_MODEL_PATH is None:
    # Look for best.keras to export
    keras_file = RUN_DIR / 'best.keras'
    if not keras_file.exists():
        keras_candidates = list(RUN_DIR.glob('*.keras'))
        if keras_candidates:
            keras_file = keras_candidates[0]
        else:
            raise FileNotFoundError(f"No .onnx model or .keras checkpoint found in {RUN_DIR}!")
    
    print(f"Exporting {keras_file.name} to ONNX format using fast-plate-ocr export...")
    !fast-plate-ocr export --format onnx --plate-config-file "{str(PLATE_CONFIG_PATH)}" --simplify --model "{str(keras_file)}"
    
    # Check where exported best.onnx was created
    if (RUN_DIR / 'best.onnx').exists():
        ONNX_MODEL_PATH = RUN_DIR / 'best.onnx'
    elif Path('best.onnx').exists():
        shutil.copy2('best.onnx', RUN_DIR / 'best.onnx')
        ONNX_MODEL_PATH = RUN_DIR / 'best.onnx'
    else:
        raise FileNotFoundError(f"fast-plate-ocr export completed but best.onnx was not found!")

print(f"Using ONNX Model: {ONNX_MODEL_PATH} ({ONNX_MODEL_PATH.stat().st_size / (1024*1024):.2f} MB)")

# 4. Initialize LicensePlateRecognizer
recognizer = LicensePlateRecognizer(
    onnx_model_path=str(ONNX_MODEL_PATH),
    plate_config_path=str(PLATE_CONFIG_PATH),
    device='auto',
)
print(f"[OK] LicensePlateRecognizer initialized successfully with providers: {recognizer.providers}!")


## Step 3: SANITY CHECK Before Any Metric: Verify Plate Text Extraction

### 🚨 Why This Check is Essential
Before calculating any dataset-wide metrics, we run the model on **10 validation crops** and verify that:
1. **Prediction Object Unwrapping**: `recognizer.run()` returns a `list[PlatePrediction]`. We extract `preds[0].plate` to get the decoded plate text.
2. **No Object Wrapper Repr**: The extracted text must **not** contain `"PLATEPREDICTION"` (the bug that broke exact match in earlier runs).
3. **Format Integrity**: The predicted string contains only uppercase letters and digits (`A-Z`, `0-9`) after removing library padding.
4. **Length Constraint**: The string length is strictly **< 13 characters** (Indian plates are 9–11 characters).

If any assertion fails, this cell immediately halts with a clear error message.

In [ ]:
# Step 3: Sanity check on 10 validation crops
OCR_DIR = DATA_DIR / 'processed' / 'ocr' if (DATA_DIR / 'processed' / 'ocr').exists() else DATA_DIR / 'ocr'
val_samples = df_real[df_real['split'] == 'val'].head(10).copy().reset_index(drop=True)

print("=" * 85)
print("=== SANITY CHECK: 10 Validation Crops (Verifying Plate Text Extraction) ===")
print("=" * 85)

sanity_rows = []
for idx, row in val_samples.iterrows():
    img_path = OCR_DIR / row['path']
    if not img_path.exists():
        raise FileNotFoundError(f"Sanity check error: Image not found at {img_path}!")
    
    # Execute inference
    preds = recognizer.run(str(img_path))
    
    # PROPER EXTRACTION: recognizer.run returns a list of PlatePrediction objects
    if isinstance(preds, list) and len(preds) > 0:
        pred_item = preds[0]
        raw_text = pred_item.plate if hasattr(pred_item, 'plate') else str(pred_item)
    elif hasattr(preds, 'plate'):
        raw_text = preds.plate
    else:
        raw_text = str(preds)
    
    # Clean library padding character (e.g. '_') and keep only uppercase A-Z and 0-9
    cleaned_pred = re.sub(r'[^A-Z0-9]', '', raw_text.upper()).strip()
    
    # Print comparison
    print(f"Crop {idx+1:02d} | True: {row['text']:<12} | Predicted: {cleaned_pred:<12} (Raw Output: {repr(raw_text)})")
    
    # STRICT ASSERTIONS
    # Assertion 1: Must not contain object wrapper string
    if "PLATEPREDICTION" in cleaned_pred:
        raise AssertionError(
            f"SANITY CHECK FAILED on crop {idx+1}: Extracted string contains 'PLATEPREDICTION'!\n"
            f"The PlatePrediction object repr was converted to string instead of accessing the `.plate` attribute.\n"
            f"Raw value received: {raw_text}"
        )
    
    # Assertion 2: Must contain only A-Z and 0-9
    if not re.fullmatch(r'^[A-Z0-9]*$', cleaned_pred):
        raise AssertionError(
            f"SANITY CHECK FAILED on crop {idx+1}: Predicted string '{cleaned_pred}' contains illegal characters!\n"
            f"Expected only alphanumeric characters [A-Z0-9]."
        )
    
    # Assertion 3: Length must be strictly under 13 characters
    if len(cleaned_pred) >= 13:
        raise AssertionError(
            f"SANITY CHECK FAILED on crop {idx+1}: Predicted string '{cleaned_pred}' has length {len(cleaned_pred)} (>= 13)!\n"
            f"License plate strings must be under 13 characters."
        )
    
    sanity_rows.append({
        "Crop": idx + 1,
        "True Plate": row['text'],
        "Predicted Plate": cleaned_pred,
        "Exact Match": row['text'] == cleaned_pred
    })

print("\n" + "=" * 85)
print("[OK] SANITY CHECK PASSED SUCCESSFULLY!")
print("  - All 10 validation predictions properly unwrapped from `PlatePrediction.plate`.")
print("  - Extracted strings contain exclusively valid characters [A-Z0-9].")
print("  - All extracted strings have length < 13.")
print("=" * 85)
display(pd.DataFrame(sanity_rows))


## Step 4: Evaluate on VAL: Raw vs Indian Syntax Post-Processing

We now evaluate the fine-tuned model across the entire **validation split** ($N = 167$) under two regimes:
1. **Raw Model Predictions**: Cleaned alphanumeric outputs directly from the fine-tuned model.
2. **Indian Syntax Post-Processing**: Re-implements our syntax repair engine from `ocr_runs/postprocess_eval`:
   - **Length Gating**: Applied only to plates of length 9, 10, or 11 characters (`AA 00 AA 0000`).
   - **Positional Confusable Swaps**:
     - State/Series letter positions: `0->O, 1->I, 2->Z, 5->S, 8->B, 6->G`
     - RTO/Number digit positions: `O,Q,D->0, I,L->1, Z->2, S->5, B->8, 6->G`
   - **RTO State/UT Validation**: Validates the 2-letter state prefix against the 40 official Indian RTO codes; repairs if edit distance is exactly 1 based on frequency priority.

We display a comparison table and automatically pick the superior variant on VAL for our single evaluation on TEST.

In [ ]:
# Step 4: Evaluate on VAL with and without Indian Syntax Post-Processing
import numpy as np

# --- Indian License Plate Post-Processing Rules ---
CONFUSABLE_LETTER_TO_DIGIT = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'Z': '2', 'S': '5', 'B': '8', 'G': '6'}
CONFUSABLE_DIGIT_TO_LETTER = {'0': 'O', '1': 'I', '2': 'Z', '5': 'S', '8': 'B', '6': 'G'}
VALID_INDIAN_RTO_CODES = sorted([
    "AN", "AP", "AR", "AS", "BH", "BR", "CG", "CH", "DD", "DL",
    "DN", "GA", "GJ", "HP", "HR", "JH", "JK", "KA", "KL", "LA",
    "LD", "MH", "ML", "MN", "MP", "MZ", "NL", "OD", "OR", "PB",
    "PY", "RJ", "SK", "TN", "TR", "TS", "UA", "UK", "UP", "WB"
])
RTO_PRIORITY = ["MH", "HR", "TN", "KL", "DL", "KA", "GJ", "AP", "PB", "ML", "UP", "JK", "WB", "RJ", "AS", "HP", "CG", "TS", "PY", "JH", "OD", "BR", "MP", "GA", "SK", "CH", "NL", "TR", "MN", "MZ", "AR", "UA", "UK", "AN", "DD", "DN", "LD", "LA", "OR", "BH"]

def levenshtein_dist(s1: str, s2: str) -> int:
    """Calculates Levenshtein edit distance between two strings."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]

def calc_char_acc(s1: str, s2: str) -> float:
    """Calculates character accuracy: 1 - (Levenshtein distance / max length)."""
    max_len = max(len(s1), len(s2))
    if max_len == 0: return 1.0
    return max(0.0, 1.0 - (levenshtein_dist(s1, s2) / float(max_len)))

def postprocess_indian_plate(raw_pred: str) -> str:
    """Applies position-based confusable swaps and RTO code repair to Indian plates."""
    s = re.sub(r'[^A-Z0-9]', '', str(raw_pred).upper()).strip()
    n = len(s)
    if n not in (9, 10, 11):
        return s
    chars = list(s)
    mid_letters = n - 8
    # 1. State Code (Indices 0, 1) -> Expected: LETTERS
    for i in (0, 1):
        if chars[i] in CONFUSABLE_DIGIT_TO_LETTER: chars[i] = CONFUSABLE_DIGIT_TO_LETTER[chars[i]]
    # 2. District Code (Indices 2, 3) -> Expected: DIGITS
    for i in (2, 3):
        if chars[i] in CONFUSABLE_LETTER_TO_DIGIT: chars[i] = CONFUSABLE_LETTER_TO_DIGIT[chars[i]]
    # 3. Series Code (Indices 4 to 3 + mid_letters) -> Expected: LETTERS
    for i in range(4, 4 + mid_letters):
        if chars[i] in CONFUSABLE_DIGIT_TO_LETTER: chars[i] = CONFUSABLE_DIGIT_TO_LETTER[chars[i]]
    # 4. Number (Indices 4 + mid_letters to n) -> Expected: DIGITS
    for i in range(4 + mid_letters, n):
        if chars[i] in CONFUSABLE_LETTER_TO_DIGIT: chars[i] = CONFUSABLE_LETTER_TO_DIGIT[chars[i]]
    # 5. Check first two letters against valid Indian RTO state/UT codes
    prefix = "".join(chars[:2])
    if prefix not in VALID_INDIAN_RTO_CODES:
        dist_1 = [c for c in VALID_INDIAN_RTO_CODES if levenshtein_dist(prefix, c) == 1]
        if dist_1:
            best_c = min(dist_1, key=lambda c: RTO_PRIORITY.index(c) if c in RTO_PRIORITY else 999)
            chars[0], chars[1] = best_c[0], best_c[1]
    return "".join(chars)

# Evaluate on all VAL crops
val_df = df_real[df_real['split'] == 'val'].copy().reset_index(drop=True)
val_raw_preds = []
for _, row in val_df.iterrows():
    img_p = OCR_DIR / row['path']
    res = recognizer.run(str(img_p))
    t = res[0].plate if (isinstance(res, list) and len(res) > 0) else getattr(res, 'plate', str(res))
    val_raw_preds.append(re.sub(r'[^A-Z0-9]', '', t.upper()).strip())

val_df['pred_raw'] = val_raw_preds
val_df['pred_pp'] = [postprocess_indian_plate(p) for p in val_raw_preds]

val_raw_exact = float((val_df['text'] == val_df['pred_raw']).mean() * 100.0)
val_raw_char = float(np.mean([calc_char_acc(t, p) for t, p in zip(val_df['text'], val_df['pred_raw'])]) * 100.0)
val_pp_exact = float((val_df['text'] == val_df['pred_pp']).mean() * 100.0)
val_pp_char = float(np.mean([calc_char_acc(t, p) for t, p in zip(val_df['text'], val_df['pred_pp'])]) * 100.0)

val_summary_df = pd.DataFrame([
    {"Evaluation Variant": "Fine-Tuned Model (Raw Predictions)", "Crops": len(val_df), "Full-Plate Exact Match (%)": f"{val_raw_exact:.2f}%", "Character Accuracy (%)": f"{val_raw_char:.2f}%"},
    {"Evaluation Variant": "Fine-Tuned Model + Syntax Post-Processing", "Crops": len(val_df), "Full-Plate Exact Match (%)": f"{val_pp_exact:.2f}%", "Character Accuracy (%)": f"{val_pp_char:.2f}%"},
])

print("=" * 80)
print("=== Validation Split Evaluation: Raw vs Syntax Post-Processing ===")
print("=" * 80)
display(val_summary_df)

# Choose superior variant for held-out TEST evaluation
if val_pp_exact >= val_raw_exact:
    chosen_transform = postprocess_indian_plate
    chosen_variant_name = "Fine-Tuned + Syntax Post-Processing"
    print(f"\n[DECISION] Syntax Post-Processing achieved {val_pp_exact:.2f}% >= {val_raw_exact:.2f}% on VAL. Selected: '{chosen_variant_name}' for TEST split.")
else:
    chosen_transform = lambda x: x
    chosen_variant_name = "Fine-Tuned Raw Predictions"
    print(f"\n[DECISION] Raw Predictions achieved {val_raw_exact:.2f}% > {val_pp_exact:.2f}% on VAL. Selected: '{chosen_variant_name}' for TEST split.")


## Step 5: Run Held-Out TEST Split ONCE with the Chosen Variant

We now execute our single, held-out evaluation on the **real TEST split** ($N = 164$ crops) using the superior variant chosen from VAL.

### 📊 Required Evaluation Slices:
1. **All Test Crops** ($N = 164$)
2. **Unseen-Text Only** (`text_seen_in_train == False`, $N = 143$): Measures true generalization to vehicle plates never encountered during training.
3. **One Crop per Unique Plate** ($N = 140$): Evaluates unique physical vehicles without sequence replication bias.
4. **Without Suspicious Rows** (`suspicious == False`, $N = 158$): Approved standard Indian HSRP numbers.
5. **With Suspicious Rows** (All test rows).

All per-crop predictions and summary tables are saved to `/content/drive/MyDrive/anpr/ocr_runs/finetune/eval_fixed/`.

In [ ]:
# Step 5: Single held-out evaluation on real TEST split
test_df = df_real[df_real['split'] == 'test'].copy().reset_index(drop=True)

test_raw_preds = []
for _, row in test_df.iterrows():
    img_p = OCR_DIR / row['path']
    res = recognizer.run(str(img_p))
    t = res[0].plate if (isinstance(res, list) and len(res) > 0) else getattr(res, 'plate', str(res))
    test_raw_preds.append(re.sub(r'[^A-Z0-9]', '', t.upper()).strip())

test_df['pred_raw'] = test_raw_preds
test_df['pred_final'] = [chosen_transform(p) for p in test_raw_preds]
test_df['exact_match'] = test_df['text'] == test_df['pred_final']
test_df['char_acc'] = [calc_char_acc(t, p) for t, p in zip(test_df['text'], test_df['pred_final'])]
test_df['edit_dist'] = [levenshtein_dist(t, p) for t, p in zip(test_df['text'], test_df['pred_final'])]

# Evaluate all required reporting slices
test_slices = {
    "(a) All Test Crops": test_df,
    "(b) Unseen Text Only (text_seen_in_train == False)": test_df[~test_df['text_seen_in_train']],
    "(c) Unique Plate Text (1 crop per plate)": test_df.drop_duplicates(subset=['text']),
    "(d) Without Suspicious Rows (suspicious == False)": test_df[~test_df['suspicious']],
    "(e) With Suspicious Rows (All Test Rows)": test_df,
}

slice_rows = []
for s_name, sdf in test_slices.items():
    cnt = len(sdf)
    exact_pct = float(sdf['exact_match'].mean() * 100.0) if cnt > 0 else 0.0
    char_pct = float(sdf['char_acc'].mean() * 100.0) if cnt > 0 else 0.0
    slice_rows.append({
        "Evaluation Slice": s_name,
        "Crops": cnt,
        "Full-Plate Exact Match (%)": f"{exact_pct:.2f}%",
        "Character Accuracy (%)": f"{char_pct:.2f}%",
        "_exact": exact_pct,
        "_char": char_pct,
    })

test_summary_df = pd.DataFrame(slice_rows)

print("=" * 85)
print(f"=== Held-Out Real TEST Split Evaluation ({chosen_variant_name}) ===")
print("=" * 85)
display(test_summary_df[["Evaluation Slice", "Crops", "Full-Plate Exact Match (%)", "Character Accuracy (%)"]])

# Setup destination directory: .../finetune/eval_fixed/
EVAL_OUTPUT_DIR = Path('/content/drive/MyDrive/anpr/ocr_runs/finetune/eval_fixed')
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_EVAL_DIR = RUN_DIR / 'eval_fixed'
RUN_EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Save per-crop predictions
test_df.to_csv(EVAL_OUTPUT_DIR / 'test_predictions.csv', index=False)
test_df.to_csv(RUN_EVAL_DIR / 'test_predictions.csv', index=False)
test_summary_df.to_csv(EVAL_OUTPUT_DIR / 'test_slices_summary.csv', index=False)
val_summary_df.to_csv(EVAL_OUTPUT_DIR / 'val_summary.csv', index=False)
print(f"\n[SAVED] Per-crop test predictions saved to: {EVAL_OUTPUT_DIR / 'test_predictions.csv'}")


## Step 6: Quality Tiers on TEST by Crop Height

In real deployment, plate recognition accuracy is strongly correlated with camera resolution and vehicle distance. We group test crops into **three image resolution tiers** based on crop pixel height:
1. **Low Resolution** (< 24 px height)
2. **Medium Resolution** (24–40 px height)
3. **High Resolution** (> 40 px height)

In [ ]:
# Step 6: Quality tiers on TEST by crop height
from PIL import Image

crop_heights = []
for _, row in test_df.iterrows():
    img_p = OCR_DIR / row['path']
    with Image.open(img_p) as im:
        crop_heights.append(im.size[1])

test_df['crop_height'] = crop_heights

def assign_tier(h):
    if h < 24:
        return "< 24 px (Low Res)"
    elif h <= 40:
        return "24-40 px (Medium Res)"
    else:
        return "> 40 px (High Res)"

test_df['quality_tier'] = test_df['crop_height'].apply(assign_tier)

tier_order = ["< 24 px (Low Res)", "24-40 px (Medium Res)", "> 40 px (High Res)"]
tier_rows = []
for tier in tier_order:
    tdf = test_df[test_df['quality_tier'] == tier]
    cnt = len(tdf)
    exact_pct = float(tdf['exact_match'].mean() * 100.0) if cnt > 0 else 0.0
    char_pct = float(tdf['char_acc'].mean() * 100.0) if cnt > 0 else 0.0
    tier_rows.append({
        "Quality Tier (Crop Height)": tier,
        "Crops": cnt,
        "Full-Plate Exact Match (%)": f"{exact_pct:.2f}%",
        "Character Accuracy (%)": f"{char_pct:.2f}%",
        "_exact": exact_pct,
        "_char": char_pct
    })

tier_df = pd.DataFrame(tier_rows)
print("=" * 80)
print("=== Performance Breakdown by Image Resolution Tier (Crop Height) ===")
print("=" * 80)
display(tier_df[["Quality Tier (Crop Height)", "Crops", "Full-Plate Exact Match (%)", "Character Accuracy (%)"]])
tier_df.to_csv(EVAL_OUTPUT_DIR / 'quality_tiers.csv', index=False)


## Step 7: Multi-Frame Voting Estimate on TEST Groups

When an ANPR camera monitors traffic, each vehicle produces a multi-frame track across several video frames. We simulate video track fusion using **temporal multi-frame voting**:
- Groups test crops by video/sequence origin using `split_groups.json` or filename prefixes.
- Performs **character-level majority voting** across predictions in the track.

> ⚠️ **Important Notice**: This is an **empirical estimate** demonstrating potential gains from multi-frame camera persistence.

In [ ]:
# Step 7: Multi-frame temporal voting estimate on sequence groups
from collections import Counter
import json

def extract_sequence_group(path_str: str) -> str:
    """Extract sequence / video group ID from crop filename or relative path."""
    fname = Path(path_str).name
    m1 = re.match(r'^(video\d+)', fname, re.IGNORECASE)
    if m1: return f"vid_{m1.group(1).lower()}"
    m2 = re.match(r'^(car-[a-zA-Z0-9]+-[A-Z0-9]+)_\d+', fname, re.IGNORECASE)
    if m2: return f"vid_{m2.group(1).lower()}"
    stem = Path(path_str).stem
    if '___' in stem:
        return stem.split('___')[0]
    m3 = re.match(r'^([A-Za-z0-9_-]+)_\d+', fname)
    if m3: return f"vid_{m3.group(1).lower()}"
    return stem

# Load split_groups.json if available
sg_path = DATA_DIR / 'processed' / 'split_groups.json' if (DATA_DIR / 'processed' / 'split_groups.json').exists() else DATA_DIR / 'split_groups.json'
group_map = {}
if sg_path.exists():
    try:
        with open(sg_path, 'r', encoding='utf-8') as f:
            sg_data = json.load(f)
            group_map = sg_data.get('image_groups', {})
    except Exception as e:
        print(f"Notice: split_groups.json parsing note: {e}")

def resolve_group(row_path: str) -> str:
    norm_p = str(row_path).replace('/', '\\')
    for k, v in group_map.items():
        if k in norm_p or Path(k).name == Path(row_path).name:
            return v
    return extract_sequence_group(row_path)

test_df['group_id'] = [resolve_group(p) for p in test_df['path']]
unique_groups = test_df['group_id'].unique()
total_groups_count = len(unique_groups)

def multi_frame_vote(preds: list[str]) -> str:
    """Combine multi-frame predictions via character-level or string-level majority voting."""
    if not preds: return ""
    lengths = [len(p) for p in preds]
    if len(set(lengths)) == 1 and lengths[0] > 0:
        L = lengths[0]
        voted = []
        for i in range(L):
            char_votes = Counter([p[i] for p in preds])
            voted.append(char_votes.most_common(1)[0][0])
        return "".join(voted)
    return Counter(preds).most_common(1)[0][0]

group_records = []
for gid, gdf in test_df.groupby('group_id'):
    gt_label = Counter(gdf['text']).most_common(1)[0][0]
    voted_pred = multi_frame_vote(list(gdf['pred_final']))
    is_match = (voted_pred == gt_label)
    group_records.append({
        "group_id": gid,
        "crops_in_group": len(gdf),
        "true_label": gt_label,
        "voted_prediction": voted_pred,
        "exact_match": is_match,
        "char_acc": calc_char_acc(gt_label, voted_pred),
    })

group_df = pd.DataFrame(group_records)
group_exact_pct = float(group_df['exact_match'].mean() * 100.0)
group_char_pct = float(group_df['char_acc'].mean() * 100.0)
crop_exact_pct = float(test_df['exact_match'].mean() * 100.0)
crop_char_pct = float(test_df['char_acc'].mean() * 100.0)

voting_comparison_df = pd.DataFrame([
    {"Evaluation Level": "Single-Crop (Per-Frame)", "Units Evaluated": f"{len(test_df)} crops", "Full-Plate Exact Match (%)": f"{crop_exact_pct:.2f}%", "Character Accuracy (%)": f"{crop_char_pct:.2f}%"},
    {"Evaluation Level": "Multi-Frame Sequence Voting (Estimate)", "Units Evaluated": f"{total_groups_count} sequence groups", "Full-Plate Exact Match (%)": f"{group_exact_pct:.2f}%", "Character Accuracy (%)": f"{group_char_pct:.2f}%"},
])

print("=" * 85)
print("=== Multi-Frame Temporal Voting Evaluation (Empirical Estimate) ===")
print("=" * 85)
print(f"Total Sequence Groups: {total_groups_count} (across {len(test_df)} test crops)")
display(voting_comparison_df)
print("\n* Notice: Multi-frame sequence voting is an empirical estimate showing gains from multi-view persistence.")
voting_comparison_df.to_csv(EVAL_OUTPUT_DIR / 'voting_comparison.csv', index=False)


## Step 8: Error Diagnostics: Top-15 Character Confusions & 30 Worst Errors

To understand remaining failure modes, we perform character-level alignment and inspect failure cases:
1. **Levenshtein Alignment Matrix**: Aligns true and predicted character sequences to identify the top 15 substitution confusions ($C_{\text{true}} \rightarrow C_{\text{pred}}$).
2. **30 Worst Remaining Errors**: Displays the failure cases with the highest edit distance in a summary table and a **6×5 image grid** showing the actual plate crops annotated with true and predicted labels.

In [ ]:
# Step 8: Top-15 character confusions and 30 worst remaining errors
import matplotlib.pyplot as plt
from collections import Counter

def align_strings(s1: str, s2: str):
    """Needleman-Wunsch / Levenshtein alignment between true and predicted text."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
            
    i, j = m, n
    alignment = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if s1[i - 1] == s2[j - 1] else 1):
            alignment.append((s1[i - 1], s2[j - 1]))
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            alignment.append((s1[i - 1], '-'))
            i -= 1
        else:
            alignment.append(('-', s2[j - 1]))
            j -= 1
    return alignment[::-1]

# Count substitutions
confusions = Counter()
for _, row in test_df.iterrows():
    t, p = row['text'], row['pred_final']
    for c_true, c_pred in align_strings(t, p):
        if c_true != c_pred and c_true != '-' and c_pred != '-':
            confusions[(c_true, c_pred)] += 1

top_15_confusions = confusions.most_common(15)
confusion_table = pd.DataFrame([
    {"Rank": i, "True Char": c[0][0], "Pred Char": c[0][1], "Substitution": f"{c[0][0]} -> {c[0][1]}", "Count": c[1]}
    for i, c in enumerate(top_15_confusions, 1)
])

print("=" * 75)
print("=== Top-15 Character Confusions on Held-Out Real Test Split ===")
print("=" * 75)
display(confusion_table)
confusion_table.to_csv(EVAL_OUTPUT_DIR / 'top_confusions.csv', index=False)

# 30 Worst Remaining Errors
test_errors_df = test_df[~test_df['exact_match']].sort_values(by='edit_dist', ascending=False)
print(f"\nRemaining Test Errors: {len(test_errors_df)} / {len(test_df)} ({len(test_errors_df)/len(test_df)*100:.1f}% error rate)")

worst_30 = test_errors_df.head(30)
print("\n--- Table of 30 Worst Remaining Test Errors ---")
display(worst_30[['path', 'text', 'pred_final', 'edit_dist', 'suspicious']].reset_index(drop=True))
worst_30.to_csv(EVAL_OUTPUT_DIR / 'worst_30_errors.csv', index=False)

# Plot 30 worst error crops
fig, axes = plt.subplots(6, 5, figsize=(20, 16))
fig.patch.set_facecolor('#0f172a')

for idx, ax in enumerate(axes.flat):
    if idx < len(worst_30):
        row = worst_30.iloc[idx]
        img_p = OCR_DIR / row['path']
        with Image.open(img_p) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(
            f"True: {row['text']}\nPred: {row['pred_final']} (d={row['edit_dist']})",
            fontsize=8, color='#f87171', fontweight='bold', pad=2
        )
    ax.axis('off')

plt.suptitle("30 Worst Remaining Errors on Held-Out Test Split (Fine-Tuned Model)", fontsize=15, color='#f8fafc', y=0.98)
plt.tight_layout()
plt.savefig(EVAL_OUTPUT_DIR / 'worst_30_errors.png', dpi=130, facecolor=fig.get_facecolor())
plt.savefig(RUN_EVAL_DIR / 'worst_30_errors.png', dpi=130, facecolor=fig.get_facecolor())
plt.show()


## Step 9: Save Deliverables, Honest Benchmark Comparison Table & Dynamic Summary

In our final step, we:
1. **Save `metrics.json`**: All numpy types are cleanly converted to standard Python native types (`float`, `int`, `list`, `dict`).
2. **Honest Comparison Table**: Compares our fixed fine-tuned evaluation against **ONLY measured numbers from our project's benchmark notebooks** on unseen test plates:
   - **Scratch CRNN (PyTorch, 2-Stage Train)**: 29.37% full-plate / 81.25% character (`colab_train_ocr.ipynb`)
   - **Pretrained `fast-plate-ocr` (Zero-Shot)**: 25.17% full-plate / 79.32% character (`colab_eval_pretrained_ocr.ipynb`)
   - **Pretrained + Syntax Post-Processing**: 37.06% full-plate / 81.10% character (`colab_ocr_postprocess.ipynb`)
   - **Fine-Tuned `cct-s-v2` (Fixed Evaluation)**: Measured directly from the unseen test slice in this notebook.
3. **Dynamic Honest Summary**: Every single conclusion and comparison sentence is computed from the results, with zero hard-coded assertions.

In [ ]:
# Step 9: Save metrics.json, generate honest benchmark comparison table and dynamic summary
import json

# Unseen test subset metrics
unseen_test = test_df[~test_df['text_seen_in_train']]
ft_unseen_exact = float(unseen_test['exact_match'].mean() * 100.0)
ft_unseen_char = float(unseen_test['char_acc'].mean() * 100.0)

all_test_exact = float(test_df['exact_match'].mean() * 100.0)
all_test_char = float(test_df['char_acc'].mean() * 100.0)
clean_test_exact = float(test_df[~test_df['suspicious']]['exact_match'].mean() * 100.0)
clean_test_char = float(test_df[~test_df['suspicious']]['char_acc'].mean() * 100.0)

# 1. Save metrics.json with clean standard Python native types (no numpy types)
metrics_dict = {
    "model_name": "cct_s_v2_finetuned_indian",
    "evaluation_variant": chosen_variant_name,
    "run_directory": str(RUN_DIR),
    "measured_benchmarks_unseen_test": {
        "scratch_crnn": {"full_plate_exact_pct": 29.37, "char_acc_pct": 81.25, "source": "colab_train_ocr.ipynb"},
        "pretrained_zeroshot": {"full_plate_exact_pct": 25.17, "char_acc_pct": 79.32, "source": "colab_eval_pretrained_ocr.ipynb"},
        "pretrained_postprocessed": {"full_plate_exact_pct": 37.06, "char_acc_pct": 81.10, "source": "colab_ocr_postprocess.ipynb"},
        "finetuned_re_evaluated": {"full_plate_exact_pct": float(ft_unseen_exact), "char_acc_pct": float(ft_unseen_char), "source": "colab_eval_finetuned.ipynb"}
    },
    "test_evaluations": {
        "all_crops_count": int(len(test_df)),
        "all_crops_exact_pct": float(all_test_exact),
        "all_crops_char_acc_pct": float(all_test_char),
        "unseen_text_count": int(len(unseen_test)),
        "unseen_text_exact_pct": float(ft_unseen_exact),
        "unseen_text_char_acc_pct": float(ft_unseen_char),
        "clean_plates_count": int(len(test_df[~test_df['suspicious']])),
        "clean_plates_exact_pct": float(clean_test_exact),
        "clean_plates_char_acc_pct": float(clean_test_char),
        "unique_plates_count": int(len(test_df.drop_duplicates(subset=['text']))),
        "unique_plates_exact_pct": float(test_df.drop_duplicates(subset=['text'])['exact_match'].mean() * 100.0),
        "multi_frame_voting_exact_pct": float(group_exact_pct),
        "multi_frame_voting_char_pct": float(group_char_pct),
        "multi_frame_groups_count": int(total_groups_count),
    },
    "top_confusions": [
        {"true": c[0][0], "pred": c[0][1], "count": int(c[1])}
        for c in top_15_confusions
    ]
}

with open(EVAL_OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2)
with open(RUN_EVAL_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2)
print(f"[SAVED] metrics.json to {EVAL_OUTPUT_DIR / 'metrics.json'}")

# 2. Build Honest Benchmark Comparison Table (ONLY Measured Numbers)
comparison_table = pd.DataFrame([
    {"Approach": "1. Scratch CRNN (PyTorch, 2-Stage Train)", "Test Unseen Exact (%)": "29.37%", "Test Unseen Char Acc (%)": "81.25%", "Source": "colab_train_ocr.ipynb"},
    {"Approach": "2. Pretrained fast-plate-ocr (Zero-Shot)", "Test Unseen Exact (%)": "25.17%", "Test Unseen Char Acc (%)": "79.32%", "Source": "colab_eval_pretrained_ocr.ipynb"},
    {"Approach": "3. Pretrained + Syntax Post-Processing", "Test Unseen Exact (%)": "37.06%", "Test Unseen Char Acc (%)": "81.10%", "Source": "colab_ocr_postprocess.ipynb"},
    {"Approach": "4. Fine-Tuned cct-s-v2 (Fixed Re-Evaluation)", "Test Unseen Exact (%)": f"{ft_unseen_exact:.2f}%", "Test Unseen Char Acc (%)": f"{ft_unseen_char:.2f}%", "Source": "colab_eval_finetuned.ipynb"},
])

print("\n" + "=" * 85)
print("=== Honest Benchmark Comparison on Held-Out Unseen Real Test Plates (ONLY Measured Numbers) ===")
print("=" * 85)
display(comparison_table)
comparison_table.to_csv(EVAL_OUTPUT_DIR / 'model_comparison.csv', index=False)
comparison_table.to_csv(RUN_EVAL_DIR / 'model_comparison.csv', index=False)

# 3. Compute dynamic honest summary with zero hard-coded assertions
diff_zeroshot = ft_unseen_exact - 25.17
diff_scratch = ft_unseen_exact - 29.37
diff_postprocess = ft_unseen_exact - 37.06
diff_char_zeroshot = ft_unseen_char - 79.32
voting_gain = group_exact_pct - all_test_exact

high_res_exact = tier_df.loc[tier_df['Quality Tier (Crop Height)'] == '> 40 px (High Res)', '_exact'].values[0]
low_res_exact = tier_df.loc[tier_df['Quality Tier (Crop Height)'] == '< 24 px (Low Res)', '_exact'].values[0]

top_conf_desc = f"`{top_15_confusions[0][0][0]} -> {top_15_confusions[0][0][1]}` ({top_15_confusions[0][1]} occurrences)" if top_15_confusions else "None"

summary_markdown = f"""
### 📊 Re-Evaluation Findings & Comparative Benchmark Summary

- **Model Evaluated**: Fine-tuned `cct-s-v2-global-model` loaded from `{RUN_DIR.name}` via ONNX Runtime.
- **Evaluation Strategy Selected on VAL**: `{chosen_variant_name}` (VAL Exact: {val_pp_exact if chosen_transform == postprocess_indian_plate else val_raw_exact:.2f}%).
- **Test Unseen Generalization**: On vehicle plate numbers never seen in training (N = {len(unseen_test)}), the model achieved **{ft_unseen_exact:.2f}%** full-plate exact match and **{ft_unseen_char:.2f}%** character accuracy.
- **Measured Comparisons Against Prior Project Milestones**:
  - Relative to pretrained zero-shot (25.17% exact, 79.32% char), fine-tuning changed exact match by **{diff_zeroshot:+.2f}%** and character accuracy by **{diff_char_zeroshot:+.2f}%**.
  - Relative to scratch CRNN (29.37% exact, 81.25% char), fine-tuning changed exact match by **{diff_scratch:+.2f}%**.
  - Relative to pretrained + rule post-processing (37.06% exact, 81.10% char), fine-tuning changed exact match by **{diff_postprocess:+.2f}%**.
- **Resolution Impact**: Higher-resolution crops (>40 px) achieved **{high_res_exact:.2f}%** exact match, compared to **{low_res_exact:.2f}%** on low-resolution crops (<24 px).
- **Multi-Frame Video Temporal Fusion (Estimate)**: Character-level majority voting across {total_groups_count} sequence groups yielded **{group_exact_pct:.2f}%** exact match (a difference of **{voting_gain:+.2f}%** compared to single-frame per-crop accuracy of {all_test_exact:.2f}%).
- **Dominant Failure Mode**: The most frequent character substitution in remaining errors is {top_conf_desc}.
"""

print("\n" + "=" * 85)
print(summary_markdown.strip())
print("=" * 85)
print(f"\nAll evaluation deliverables, summary tables, and plots saved to:\n  - {EVAL_OUTPUT_DIR}\n  - {RUN_EVAL_DIR}")
